In [4]:
import os
import sys
import time
import torch
import torch.nn.functional as F
import numpy as np
import cv2
from collections import deque

sys.path.append('src')
from src.env import create_train_env
from src.model import PPO
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT

print("🔧 Mario Viewer - Parche Total")

WORLD, STAGE = 1, 1

# ============================================
# PARCHE UNIVERSAL STEP (ANTES de crear env)
# ============================================
class PatchedEnv:
    def __init__(self, env):
        self.env = env
    
    def step(self, action):
        """RETORNA SIEMPRE 4 VALORES: obs, reward, done, info"""
        try:
            result = self.env.step(action)
            # Manejar todos los casos posibles
            if len(result) == 5:
                obs, reward, terminated, truncated, info = result
                return obs, reward, (terminated or truncated), info
            elif len(result) == 4:
                obs, reward, done, info = result
                return obs, reward, done, info
            elif len(result) == 3:
                obs, reward, done = result
                return obs, reward, done, {}
            else:
                obs = result[0] if len(result) > 0 else np.zeros((240,256,3))
                reward = result[1] if len(result) > 1 else 0
                done = True
                info = {}
                return obs, reward, done, info
        except Exception as e:
            print(f"Step error: {e}")
            return np.zeros((240,256,3)), 0, True, {}
    
    def reset(self):
        try:
            result = self.env.reset()
            if isinstance(result, tuple) and len(result) == 2:
                return result[0]  # Solo obs
            return result
        except:
            return self.env.reset()
    
    def render(self, *args, **kwargs):
        try:
            return self.env.render(*args, **kwargs)
        except:
            pass
    
    def close(self):
        self.env.close()
    
    def __getattr__(self, name):
        return getattr(self.env, name)

print("✅ Parche universal listo")

# ============================================
# FrameProcessor
# ============================================
class FrameProcessor:
    def __init__(self):
        self.frames = deque(maxlen=4)
        self.initialized = False
    
    def reset(self):
        self.frames.clear()
        self.initialized = False
    
    def preprocess(self, frame):
        try:
            # Si es stacked (4,84,84), usar último
            if len(frame.shape) == 4:
                frame = frame[-1]
            
            # Si es RGB
            if len(frame.shape) == 3:
                gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            else:
                gray = frame
            
            resized = cv2.resize(gray, (84, 84))
            return resized.astype(np.float32) / 255.0
        except:
            return np.zeros((84, 84), dtype=np.float32)
    
    def process(self, obs):
        proc = self.preprocess(obs)
        if not self.initialized:
            for _ in range(4): self.frames.append(proc)
            self.initialized = True
        else:
            self.frames.append(proc)
        return np.stack(self.frames, axis=0)

# ============================================
# Crear ENV PARCHADO
# ============================================
print("🎮 Creando env...")
raw_env = create_train_env(WORLD, STAGE, SIMPLE_MOVEMENT, None)
env = PatchedEnv(raw_env)  # ← PARCHE AQUÍ
processor = FrameProcessor()

print("✅ Env parchado")

# ============================================
# Modelo
# ============================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PPO(4, len(SIMPLE_MOVEMENT))
model.eval()

# Carga robusta
model_files = os.listdir("trained_models")
for f in model_files:
    if f.startswith("ppo_super_mario_bros") or f.endswith(('.pt','.pth')):
        try:
            path = os.path.join("trained_models", f)
            sd = torch.load(path, map_location=device)
            model.load_state_dict(sd)
            if torch.cuda.is_available():
                model.cuda()
            print(f"✅ Modelo: {f}")
            break
        except:
            continue
else:
    print("❌ No modelo. Archivos:", model_files[:5])
    sys.exit(1)

# ============================================
# 🎮 JUGAR
# ============================================
obs = env.reset()
processor.reset()
state = processor.process(obs)
state_t = torch.FloatTensor(state).unsqueeze(0).to(device)

steps = total_reward = 0
action_names = ['NOOP','→','→+A','→+B','↓→','↑','→↑','A','←','←+A','←+B','↓←','←↑','B','↓']

print("\n🚀 ¡LISTO! Ventana en 2s...")
time.sleep(2)

print(f"🎮 JUGANDO World {WORLD}-{STAGE}")
print("="*50)

try:
    while True:
        # Acción IA
        with torch.no_grad():
            logits, _ = model(state_t)
            action = torch.argmax(F.softmax(logits, dim=1), dim=1).item()
        
        # SIEMPRE 4 valores
        obs, reward, done, info = env.step(action)
        env.render()  # 🪟 VENTANA
        
        state = processor.process(obs)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        steps += 1
        total_reward += reward
        
        if steps % 100 == 0:
            x = info.get('x_pos', 0)
            print(f"{steps:4d} | {action_names[action]:5s} | X:{x:4d} | R:{total_reward:5.1f}")
        
        if info.get('flag_get'):
            print(f"\n🏆 ¡VICTORIA! {steps} pasos")
            break
        if done:
            print(f"\n💀 Game Over {steps} pasos")
            break
        
        time.sleep(0.02)

except KeyboardInterrupt:
    print(f"\n⏹️ Stop {steps}")

finally:
    env.close()
    print("✅ Fin")

🔧 Mario Viewer - Parche Total
✅ Parche universal listo
🎮 Creando env...


/home/deynar/miniconda3/envs/mario_ppo/lib/python3.8/site-packages/gym/envs/registration.py:555: UserWarning: WARN: The environment SuperMarioBros-1-1-v0 is out of date. You should consider upgrading to version `v3`.
  logger.warn(


✅ Env parchado
✅ Modelo: ppo_super_mario_bros_1_1

🚀 ¡LISTO! Ventana en 2s...


/tmp/ipykernel_84622/1770549404.py:134: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(path, map_location=device)


🎮 JUGANDO World 1-1
Step error: not enough values to unpack (expected 5, got 4)

💀 Game Over 1 pasos
✅ Fin
